# Day 1 · Section 2: Set up and inspect your Colab environment

**Goal:** become comfortable with the notebook runtime before loading a large model. You will identify the Python process and GPU, check CUDA and memory, install and test the workshop packages, inspect the temporary filesystem, then run your first Qwen3 prompt.

This is a **standalone student notebook**. Run cells from top to bottom. CPU checks run without a GPU. Download and model cells are gated; they report a skip when their requirements are unavailable. It contains no workshop slides.

**Estimated time:** 35–60 minutes, excluding model download. **Resources:** internet access for packages and Qwen weights; a Colab GPU with about 10 GiB or more free memory for the 4B model in half precision. Actual memory use varies with the runtime and prompt.


## 2.1 · How a Colab session works

The notebook file stores your code and saved outputs. Execution happens in a separate, temporary virtual machine. Variables and installed packages belong to the current session; a runtime reset removes them. Files under `/content` are temporary unless you explicitly copy them elsewhere.

In Colab, choose **Runtime → Change runtime type → GPU**, save, and reconnect. A GPU might not be available on every account at every moment. The following cells inspect the runtime you actually received.


In [ ]:
import os, sys, platform, pathlib, subprocess, importlib.util
IN_COLAB = 'google.colab' in sys.modules
print('Colab:',IN_COLAB)
print('Python:',sys.version.split()[0])
print('Executable:',sys.executable)
print('Operating system:',platform.platform())
print('Current directory:',pathlib.Path.cwd())
print('CPU cores:',os.cpu_count())
print('Session PID:',os.getpid())


### CPU memory and disk are different from GPU memory

CPU RAM holds Python objects and downloaded model files during loading. Disk holds files. GPU VRAM holds model weights and tensors during accelerated inference. A large free disk does not imply enough VRAM.


In [ ]:
import shutil
disk=shutil.disk_usage(pathlib.Path.cwd())
print(f'Disk free: {disk.free/2**30:.1f} GiB of {disk.total/2**30:.1f} GiB')
try:
    import psutil
    ram=psutil.virtual_memory()
    print(f'CPU RAM available: {ram.available/2**30:.1f} GiB of {ram.total/2**30:.1f} GiB')
except ImportError:
    print('psutil not installed; CPU RAM figure omitted.')


### Identify the GPU independently of PyTorch

`nvidia-smi` reports the GPU model, total VRAM, current use and driver. It is a system utility, not a PyTorch test. A missing command usually means no NVIDIA GPU is attached.


In [ ]:
check=subprocess.run(['nvidia-smi'],capture_output=True,text=True,check=False) if shutil.which('nvidia-smi') else None
if check and check.returncode==0:
    print('\n'.join(check.stdout.splitlines()[:18]))
else:
    print('nvidia-smi unavailable. CPU cells remain usable; select a GPU runtime for model work.')


### Check CUDA from PyTorch

PyTorch must see the GPU before model loading. The GPU name, free VRAM and total VRAM below describe the current session. The code keeps `HAS_GPU=False` when PyTorch is absent, so the remaining CPU checks still run.


In [ ]:
try:
    import torch
    HAS_GPU=torch.cuda.is_available()
    print('PyTorch:',torch.__version__,'CUDA build:',torch.version.cuda,'GPU visible:',HAS_GPU)
    if HAS_GPU:
        free_bytes,total_bytes=torch.cuda.mem_get_info()
        print('Device:',torch.cuda.get_device_name(0))
        print(f'Free/total GPU VRAM: {free_bytes/2**30:.2f}/{total_bytes/2**30:.2f} GiB')
        print('BF16 supported:',torch.cuda.is_bf16_supported())
except ImportError:
    torch=None;HAS_GPU=False
    print('PyTorch is absent. In Colab it is normally preinstalled.')


### Run one small computation on the selected device

GPU allocation rises while a tensor is live. Releasing Python references may make memory reusable by PyTorch without immediately returning it to the operating system. This small matrix multiplication is a smoke test, not a benchmark.


In [ ]:
import time
if torch is not None:
    device=torch.device('cuda:0' if HAS_GPU else 'cpu')
    a=torch.randn((512,512),device=device)
    before=torch.cuda.memory_allocated() if HAS_GPU else 0
    start=time.perf_counter(); result=a@a
    if HAS_GPU:torch.cuda.synchronize()
    elapsed=time.perf_counter()-start
    print('Device:',result.device,'result shape:',tuple(result.shape),'elapsed:',round(elapsed,4),'s')
    if HAS_GPU:print('Allocated VRAM before/after result:',before,torch.cuda.memory_allocated(),'bytes')
    del a,result
else:print('PyTorch computation skipped.')


## 2.2 · Inspect and install Python packages

Use `%pip` or `python -m pip` in the active notebook kernel. `pip` in a different terminal can install into a different Python environment. The next cell lists versions before any change. In Colab, the install cell runs by default; locally, set `INSTALL_PACKAGES=True` if you want to install into your current environment.


In [ ]:
from importlib import metadata
packages=['numpy','torch','transformers','accelerate','sentence-transformers','faiss-cpu','pypdf','rank-bm25']
def version_of(name):
    try:return metadata.version(name)
    except metadata.PackageNotFoundError:return 'not installed'
for name in packages:print(f'{name:23} {version_of(name)}')


In [ ]:
INSTALL_PACKAGES = IN_COLAB  # local Jupyter: opt in deliberately
if INSTALL_PACKAGES:
    required=['transformers>=4.52.4,<6','accelerate','safetensors','sentence-transformers',
              'faiss-cpu','pypdf','rank-bm25']
    subprocess.check_call([sys.executable,'-m','pip','-q','install',*required])
    print('Installed workshop packages into:',sys.executable)
else:
    print('Installation skipped. Set INSTALL_PACKAGES=True to install locally.')


### Verify imports and a tiny retrieval operation

This checks package usability, not just whether a package name appears in a list. FAISS `IndexFlatIP` performs exact inner-product search; the two test vectors below are normalized, so the scores also act as cosine similarities.


In [ ]:
import numpy as np
from packaging.version import Version
print('NumPy:',np.__version__)
if importlib.util.find_spec('faiss'):
    import faiss
    vectors=np.array([[1.,0.],[0.,1.]],dtype='float32')
    index=faiss.IndexFlatIP(2);index.add(vectors)
    scores,indices=index.search(np.array([[.9,.1]],dtype='float32'),1)
    print('FAISS nearest index:',indices[0,0],'score:',round(float(scores[0,0]),3))
    assert int(indices[0,0])==0
else:print('FAISS absent; install packages above to run its smoke test.')
if importlib.util.find_spec('pypdf'):
    from pypdf import PdfReader
    print('pypdf imported; PDF parsing is used on Day 2.')
else:print('pypdf absent.')


### Check Accelerate's device selection

`Accelerator()` detects a usable device and mixed-precision setting. Creating it does not itself move a model or tensor. The model loading step later uses `device_map="auto"`.


In [ ]:
if importlib.util.find_spec('accelerate'):
    from accelerate import Accelerator
    accelerator=Accelerator()
    print('Accelerate device:',accelerator.device)
    print('Mixed precision:',accelerator.mixed_precision)
else:print('Accelerate absent; run the package installation cell.')


### Explore temporary files safely

This demonstration writes a tiny file into a temporary directory and removes it automatically. If you need a durable student copy, save the `.ipynb` file to your own drive or download it. You do not need to mount Drive for these exercises.


In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as temporary:
    demo=pathlib.Path(temporary)/'workshop_check.txt'
    demo.write_text('A Colab runtime stores files separately from notebook cells.\n')
    print('Temporary file:',demo)
    print('Read-back:',demo.read_text().strip())
print('Directory removed after the with-block:',not demo.exists())


## 2.3 · Load the Qwen3 tokenizer

The tokenizer download is much smaller than model weights and works on CPU. Its chat template inserts role boundaries and assistant-generation markers. Do not manually invent these control tokens. In Colab the download is enabled after installation; locally set `RUN_DOWNLOADS=True` if packages and network are ready.


In [ ]:
RUN_DOWNLOADS = IN_COLAB
MODEL_ID='Qwen/Qwen3-4B'
tokenizer=None
if RUN_DOWNLOADS and importlib.util.find_spec('transformers'):
    from transformers import AutoConfig, AutoTokenizer
    config=AutoConfig.from_pretrained(MODEL_ID)
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    print('Model type:',config.model_type,'hidden size:',config.hidden_size,'layers:',config.num_hidden_layers)
    print('Tokenizer base vocab:',tokenizer.vocab_size,'EOS ID:',tokenizer.eos_token_id)
else:print('Tokenizer download skipped. Enable RUN_DOWNLOADS after installing packages.')


In [ ]:
if tokenizer is not None:
    messages=[{'role':'user','content':'Explain RAG in one sentence.'}]
    formatted=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    token_ids=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,enable_thinking=False)
    print('Formatted prompt preview:\n',formatted[:400])
    print('Prompt token count:',len(token_ids),'first IDs:',token_ids[:12])
    print('First pieces:',tokenizer.convert_ids_to_tokens(token_ids[:12]))
else:print('Tokenizer cell skipped.')


## 2.3 · Decide whether the 4B model fits

FP16/BF16 uses about two bytes per parameter, before activations, attention cache and framework overhead. This gate defaults to loading only when the GPU has at least **10 GiB free**. It is a conservative classroom threshold, not a guarantee. You can lower it deliberately after checking your runtime, or use a smaller model for experiments.


In [ ]:
FREE_GPU_GIB=(torch.cuda.mem_get_info()[0]/2**30) if HAS_GPU else 0.
RUN_MODEL=HAS_GPU and tokenizer is not None and FREE_GPU_GIB>=10
print('Free GPU GiB:',round(FREE_GPU_GIB,2),'Load 4B model:',RUN_MODEL)
if HAS_GPU and not RUN_MODEL:
    print('The GPU is visible, but free memory or tokenizer setup does not meet the default gate.')


### Load weights in half precision

Do not rerun this cell repeatedly while keeping older model objects alive. `device_map="auto"` can place parts on different devices if needed; inspect the result rather than assuming everything is on `cuda:0`. Downloads are cached within the current session.


In [ ]:
model=None
if RUN_MODEL:
    from transformers import AutoModelForCausalLM
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    load_start=time.perf_counter()
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=dtype,device_map='auto')
    print('Loaded in',round(time.perf_counter()-load_start,1),'s')
    print('Model type:',type(model).__name__,'dtype:',next(model.parameters()).dtype)
    print('Device map:',getattr(model,'hf_device_map',{'model':str(model.device)}))
    print('Input embedding matrix:',tuple(model.get_input_embeddings().weight.shape))
    print('GPU allocated GiB:',round(torch.cuda.memory_allocated()/2**30,2))
else:print('Model load skipped. Earlier environment and tokenizer exercises remain complete.')


### Align input tensors and model device

The tokenizer normally returns CPU tensors. `input_ids` contains integers. The model turns them into vectors during its forward pass. The first axis is the batch (one conversation here); the second is sequence length.


In [ ]:
inputs=None
if model is not None:
    inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
        enable_thinking=False,return_dict=True,return_tensors='pt')
    print('Before:',inputs['input_ids'].shape,inputs['input_ids'].device,'axes:',inputs['input_ids'].ndim)
    inputs=inputs.to(model.device)
    print('After:',inputs['input_ids'].device,'model input device:',model.device)
    assert inputs['input_ids'].ndim==2 and inputs['input_ids'].shape[0]==1
else:print('Tensor alignment skipped.')


## 2.4 · Your first LLM program

Generate at most 80 *new* tokens, then slice the returned IDs after the prompt. Greedy decoding gives a stable baseline for this exercise; exact text can still change with model or library versions.


In [ ]:
if model is not None:
    model.eval()
    start=time.perf_counter()
    with torch.inference_mode():
        output_ids=model.generate(**inputs,max_new_tokens=80,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    if HAS_GPU:torch.cuda.synchronize()
    elapsed=time.perf_counter()-start
    prompt_length=inputs['input_ids'].shape[1]
    new_ids=output_ids[0,prompt_length:]
    response=tokenizer.decode(new_ids,skip_special_tokens=True)
    print('Prompt/new tokens:',prompt_length,len(new_ids),'elapsed s:',round(elapsed,2))
    print('Response:',response)
else:print('Generation skipped; a GPU and loaded model are needed.')


### Change one thing at a time

Try a second prompt with the same model. Then, if you wish, compare greedy decoding with sampling. `top_k` caps the number of candidates; `top_p` limits their cumulative probability mass. `max_new_tokens` is an upper bound and generation may stop earlier.


In [ ]:
if model is not None:
    second=[{'role':'user','content':'Name one benefit of retrieval in a grounded answer.'}]
    second_inputs=tokenizer.apply_chat_template(second,tokenize=True,add_generation_prompt=True,
        enable_thinking=False,return_dict=True,return_tensors='pt').to(model.device)
    torch.manual_seed(7)
    with torch.inference_mode():
        sampled=model.generate(**second_inputs,max_new_tokens=60,do_sample=True,
            temperature=.7,top_p=.8,top_k=20,pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(sampled[0,second_inputs['input_ids'].shape[1]:],skip_special_tokens=True))
else:print('Sampling comparison skipped.')


## Understanding the outputs

Fill in these answers after running the relevant cells:

1. Which Python executable is backing this notebook? Which packages were missing before setup?
2. What GPU, free VRAM and CUDA version did this session report? If no GPU appeared, which CPU exercises still worked?
3. How do CPU RAM, disk and GPU VRAM differ? Which one limits FP16 model loading?
4. Why does the input ID tensor have shape `[1, sequence_length]`?
5. Which part of the output ID tensor contains the prompt? Why did you slice it away before decoding?
6. How many prompt and new tokens did your first run use? Did it reach the maximum of 80?
7. What disappeared when you restarted a runtime? Save your notebook before trying a reset.


## Troubleshooting checklist

| Symptom | Check |
|---|---|
| `nvidia-smi` unavailable / CUDA false | Choose a GPU runtime and reconnect; rerun the environment cells. |
| Package import fails after installation | Confirm the active `sys.executable`; rerun imports or restart and rerun cells in order. |
| Model download fails | Check network access and model identifier; retry the same cell after connectivity returns. |
| CUDA out of memory | Remove other model objects, restart the runtime, shorten prompt/output, or use a smaller/quantized model. |
| Input/model device mismatch | Inspect both devices and move the tokenizer output to the model's input device. |
| Empty or odd answer | Inspect `enable_thinking`, prompt formatting, EOS handling and the generated-ID slice. |

**References:** [Qwen3-4B model card](https://huggingface.co/Qwen/Qwen3-4B) · [PyTorch CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html) · [Hugging Face Accelerate](https://huggingface.co/docs/accelerate/)
